In [ ]:
# 16.3 Chef Crew

import os
from crewai import Crew, Agent, Task

os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

international_chef = Agent(
    role="International Chef",
    goal="Create ethnic cuisine recipies that are easy to cook at home",
    backstory="""
    You are an famous chef that specializes in cuisine from countries all around the world.
    You know how to cook the most traditional dishes from all cultures but you also know how to adapt them for people to be able to cook them at home.
    """,
    verbose=True,
    allow_delegation=False, # 협업 금지
)
healthy_chef = Agent(
    role="Healthy Chef",
    goal="Turn any recipe into a healthy vegetarian recipe that is easy to cook with home ingredients.",
    backstory="""
    You are a chef specialized in healthy cooking.
    You can take any recipe and change the ingredients to make it vegetarian friendly without loosing the escense of the dish and what makes it delicious.
    """,
    verbose=True,
    allow_delegation=False,
)

normal_recipe = Task(
    description="Come up with a {dish} that serves {people} people.",
    agent=international_chef,
    expected_output="Your answer MUST have three sections, the ingredients required with their quantities, the preparation instructions and serving suggestions",
    output_file="normal_recipe.md"
)
healthy_recipe = Task(
    description="Replace the ingredients of a recipe to make it vegetarian without making it less delicious, adjust if needed.",
    agent=healthy_chef,
    expected_output="Your answer MUST have four sections, the ingredients required with their quantities, the preparation instructions, serving suggestions and an explanation of the replaced ingredients.",
    output_file="healthy_recipe.md"
)

crew = Crew(
    tasks=[
        normal_recipe,
        healthy_recipe,
    ],
    agents=[
        international_chef,
        healthy_chef,
    ],
    verbose=2
)

result = crew.kickoff(
    inputs=dict(
        dish="Greek dinner",
        people="5",
    )
)

In [ ]:
# 16.6 Pydantic Outputs
from pydantic import BaseModel
from typing import List

'''
A blog post with an introduction, at least three sub-sections of content, links to sources, a set of suggested hashtags for social media and a catchy title.
'''

class SubSection(BaseModel):
    title: str
    content: str

class BlogPost(BaseModel):
    title: str
    introduction: str
    sections: List[SubSection]
    sources: List[str]
    hashtags: List[str]

In [ ]:
# 16.5 Content Farm Crew
from crewai import Crew, Agent, Task
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

researcher = Agent(
    role="Senior Researcher",
    goal="Search the web, extract and analyze information.",
    backstory="""
    You produce the highest quality research possible.
    You use multiple sources of information and you always double check your sources to make sure they are true and up to date.
    You want to impress your coworkers with your work.
    """,
    allow_delegation=False,
    verbose=True,
    tools=[
        search_tool,
        scrape_tool
    ],
    max_iter=10,
)
editor = Agent(
    role="Senior Writer/Editor",
    goal="Write engaging blog posts.",
    backstory="""
    You write content that keeps people engaged and entertained.
    Your content is easy to read it is informative and it makes people want to share it with their friends.
    You are working for a very important client.
    """,
    verbose=True,
)

task = Task(
    description="Write a blog post about {topic}",
    agent=editor,
    expected_output="A blog post with an introduction, at least three sub-sections of content, links to sources, a set of suggested hashtags for social media and a catchy title.",
    output_file="blog_post.md",
    output_pydantic=BlogPost,
    output_json=BlogPost,
)

crew = Crew(
    agents=[researcher, editor],
    tasks=[task],
    verbose=2,
)

result = crew.kickoff(
    inputs=dict(topic="The biggest box office flops of 2024")
)

# 16.6 Pydantic Outputs
result.title
result.sections[0].title

In [ ]:
# 16.7 Async Youtuber Crew
from crewai import Crew, Agent, Task
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

researcher = Agent(
    role="Senior Researcher",
    goal="Search the web, extract and analyze information.",
    backstory="""
    You produce the highest quality research possible.
    You use multiple sources of information and you always double check your sources to make sure they are true and up to date.
    You want to impress your coworkers with your work.
    """,
    max_iter=10,
    allow_delegation=False,
    verbose=True,
    tools=[
        search_tool,
        scrape_tool,
    ],
)
marketer = Agent(
    role="Senior Marketer",
    goal="Come up with ideas that generate viral and useful content.",
    backstory="""
    You work at a marketing agency.
    You are the best at coming up with ideas to make content go viral.
    Your ideas are used for video, advertising, social media marketing, the content you produce appeals to a young audience.
    """,
    verbose=True,
)
writer = Agent(
    role="Senior Writer",
    goal="Write scripts for viral Youtube videos.",
    backstory="""
    You write scripts for videos that keep people engaged and entertained.
    Your content is easy and fun to watch, it is informative and it makes people want to share it with their friends.
    You are working for a very important client.
    """,
    verbose=True,
)

brainstorm_task = Task( # 1
    description="Come up with 5 ideas for a Youtube channel in the {industry} industry",
    agent=marketer,
    expected_output="Your answer MUST be a list of 5 ideas for a Youtube video with an explanation of what the angle of the video would be.",
    output_file="ideas_task.md",
    human_input=True,
)
selection_task = Task( # 2
    description="Select a video idea that has the highest potential of going viral.",
    agent=writer,
    expected_output="Your answer MUST include the idea that was selected as well as an explanation of why that selection was made.",
    human_input=True,
    context=[brainstorm_task],
    output_file="selection_task.md",
)
research_task = Task( # 3
    description="Do all the research required to write the script of a medium length video about the selected idea.",
    agent=researcher,
    expected_output="Your answer must have all the information a write would need to write a Youtube script.",
    async_execution=True,
    context=[selection_task],
    output_file="research_task.md",
)
competitors_task = Task( # 3
    description="Search for videos or articles in the {industry} industry that are similar to the video we are working on and suggest ways our video can be different from theirs.",
    agent=researcher,
    expected_output="Your answer must have a list of suggestions writers can follow to make sure the video is as unique and as different from competitors as possible.",
    async_execution=True,
    context=[selection_task],
    output_file="competitors_task.md",
)
inspiration_task = Task( # 3
    description="Search for videos or articles that are similar to the video idea we are working on but from other industires.",
    agent=researcher,
    expected_output="Your answer must have a list of examples of the video or articles that have a similar angle as the video we are making but that are in different industries.",
    async_execution=True,
    context=[selection_task],
    output_file="inspiration_task.md",
)
script_task = Task( # 4
    description="Write the script for a Youtube video for a channel in the {industry} industry.",
    agent=writer,
    expected_output="A script for a Youtube video with a title, an introduction, at least three sections, and an outro. Make sure to also include the prompt to generate a thumbnail for the video.",
    context=[
        selection_task,
        research_task,
        competitors_task,
        inspiration_task,
    ],
    output_file="script_task.md",
)

crew = Crew(
    agents=[researcher, marketer, writer],
    tasks=[
        brainstorm_task,
        selection_task,
        research_task,
        competitors_task,
        inspiration_task,
        script_task,
    ],
    verbose=2,
)

result = crew.kickoff(inputs=dict(industry="Hot Sauce"))

In [ ]:
# 16.8 Custom Tools
from crewai_tools import tool
import yfinance as yf

class Tools:
    @tool("One month stock price history")
    def stock_price(ticker):
        """
        Useful to get a month's worth of stock price data as CSV.
        The input of this tool should a ticker, for example AAPL, NET, TSLA.
        """
        stock = yf.Ticker(ticker)
        return stock.history(period="1mo").to_csv()
    
    @tool("Stock news URLs")
    def stock_news(ticker):
        """
        Useful to get URLs of news articles related to a stock.
        The input to this tool should be a ticker, for example AAPL, NET.
        """
        stock = yf.Ticker(ticker)
        return list(map(lambda x: x["link"], stock.news))
    
    @tool("Company's income statement")
    def income_stmt(ticker):
        """
        Useful to get income statement of a stock as CSV.
        The input to this tool should be a ticker, for example AAPL, NET.
        """
        stock = yf.Ticker(ticker)
        return stock.income_stmt.to_csv()

    @tool("Balance sheet")
    def balance_sheet(ticker):
        """
        Useful to get a balance sheet of a stock as CSV.
        The input to this tool should be a ticker, for example AAPL, NET.
        """
        stock = yf.Ticker(ticker)
        return stock.balance_sheet.to_csv()
    
    @tool("Get insider transactions")
    def insider_transactions(ticker):
        """
        Useful to get a insider transactions of a stock as CSV.
        The input to this tool should be a ticker, for example AAPL, NET.
        """
        stock = yf.Ticker(ticker)
        return stock.insider_transactions.to_csv()

In [ ]:
# 16.8 Custom Tools
from crewai import Agent
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

class Agents:
    def technical_analyst(self):
        return Agent(
            role="Technical Analysist",
            gole="analyses the movement of a stock and provides insights on trends, entry points, resistance, and support levels.",
            backstory="an expert in technical analysis, you're known for your ability to predict stock movements and trends based on historical data. You provide valuable insights to your customers.",
            verbose=True,
            tools=[Tools.stock_price],
        )

    def financial_analyst(self):
        return Agent(
            role="Financial Analyst",
            goal="Use financial statement, insider trading data, and other financial metrics to evaluate a stock's health and performance",
            backstory="You are a very experienced investment advisor who uses a combination of technical and fundamental analysis to provide strategic advice to your clients. You look at company's financial health, market sentiment, and qualitative data to make informed recommendations.",
            verbose=True,
            tools=[
                Tools.balance_sheet,
                Tools.income_stmt,
                Tools.insider_transactions,
            ],
        )

    def researcher(self):
        return Agent(
            role="Researcher",
            goal="Gathers, intepret, and summarizes vasts amount of data to provide a comprehensive overview of the sentiment and news surrounding a stock.",
            backstory="You are skilled in gathering and interpreting data from various sources to give a complete picture of a stock's sentiment and news. You read each data source carefully and extract most important information. Your insights are crucial for making informed investment decisions.",
            verbose=True,
            tools=[
                Tools.stock_news, 
                SerperDevTool(), 
                ScrapeWebsiteTool(),
            ]
        )

    def hedge_fund_manager(self):
        # buy or not
        return Agent(
            role="Hedge Fund Manager",
            goal="Manages a portfolio of stocks and makes a strategic investment decisions to maximize returns using insights from financial analysts, technical analysts, and researchers.",
            backstory="You are seasoned hedge fund manager with a proven track of record of making profitable invesment decisions. You are known for your ability to manage risk and maximize returns for your clients.",
            verbose=True,
        )

In [ ]:
# 16.9 Stock Market Crew
from crewai import Task

class Tasks:
    def research(self, agent):
        return Task(
            description="Gather and analyze the latest news and market sentiment surrounding the stock of {company}. Provide a summary of the news, and any notable shifts in the market sentiment.",
            expected_output="Your final answer MUST be a detailed summary of the news and market sentiment surrounding the stock. Include any notable shifts in market sentiment and provide insights on how these factors impact the stock's performance.",
            agent=agent,
            output_file="stock_news.md",
        )

    def technical_analysis(self, agent):
        return Task(
            description="Conduct a detailed technical analysis of the price movements of {company}'s stock and trends identify key support and resistance levels, chart patterns, and other technical indicators that could influence stock's future performance. Use historical price data and technical analysis tools to provide insights on potential entry points and price targets.",
            expected_output="Your final answer MUST be a detalied technical analysis report that includes key support and resistance levels, chart patterns, and technical indicators. Provide insights on potential entry points, price targets, and any other relevant information that could help your customer make informed investment decisions.",
            agent=agent,
            output_file="technical_analysis.md",
        )

    def financial_analysis(self, agent):
        return Task(
            description="Analyze {company}'s financial statements, insider trading data, and other financial metrics to evaluate the stock's financial health and performance. Provide insights on the company's revenue, earnings, cash flow, and other financial metrics. Use financial analysis tools and models to access the stock's valuation and growth potential.",
            expected_output="Your final answer MUST be a detailed financial analysis report that includes insights on the company's financial health, performance, and valuation. Provide an overview of the company's revenue, earnings, cash flow, and other key financial metrics. Use financial analysis tools and models to assess the stock's valuation and growth potential.",
            agent=agent,
            output_file="financial_analysis.md",
        )
    
    def investment_recommendation(self, agent, context):
        return Task(
            description="Based on the research, technical analysis, and financial analysis reports, provide a detailed investment recommendation for {company}'s stock. Include your analysis of the stock's potential risks and rewards, and provide a clear rationale for your recommendation.",
            expected_output="Your final answer MUST be a detailed investment recommendation report to BUY or SELL the stock that includes your anlaysis of the stock's potential risks and rewards. Provide a clear rationale for your recommendation based on the research, technical analysis, and financial analysis reports.",
            agent=agent,
            context=context,
            output_file="investment_recommendation.md",
        )

In [ ]:
from crewai import Crew
from crewai.process import Process
from langchain_openai import ChatOpenAI

import os

os.environ["OPENAI_API_KEY"] = "NA"

llm = ChatOpenAI(
    model="crewai-llama2",
    base_url="http://localhost:11434/v1",
)


agents = Agents()
tasks = Tasks()

researcher = agents.researcher()
technical_analyst = agents.technical_analyst()
financial_analyst = agents.financial_analyst()
hedge_fund_manager = agents.hedge_fund_manager()

research_task = tasks.research(researcher)
financial_task = tasks.financial_analysis(financial_analyst)
technical_task = tasks.technical_analysis(technical_analyst)
recommend_task = tasks.investment_recommendation(
    hedge_fund_manager, 
    [
        research_task,
        financial_task,
        technical_task,
    ],
)

crew = Crew(
    agents=[
        researcher,
        technical_analyst,
        financial_analyst,
        hedge_fund_manager,
    ],
    tasks=[
        research_task,
        financial_task,
        technical_task,
        recommend_task,
    ],
    verbose=2,
    process=Process.hierarchical,
    manager_llm=ChatOpenAI(model="gpt-4o-mini"),
    memory=True,
)

result = crew.kickoff(
    inputs=dict(company="Salesforce")
)